# 07 下個月的登革熱：用 TimesFM 預測 — 筆記本

Doc Mon 需要給地方衛生委員會的數字：Santa Luz 接下來八週該為多少登革熱病例做準備？在這本筆記本裡你會：

1. 看八年的每週登革熱病例數（`data/synthetic/ph/santa_luz_dengue_weekly.csv`，**合成資料**）；
2. 先建立兩個簡單的**基準線**——任何模型都必須跨過的門檻；
3. 在公平的**滾動起點回測**裡執行 **TimesFM 3.0**，一個零樣本的時間序列基礎模型；
4. 觀察疫情起飛時它的表現，以及雨量有沒有幫助；
5. 從 2026-W39 做出真正的預測——附上不確定性區間——並決定它能相信到什麼程度。

> **兩條路徑，同樣的數字。** 在 Google Colab（或任何已安裝 `timesfm` 的電腦）上，下面的儲存格會真的呼叫 TimesFM 3.0。建置這個網站時，不會下載 1.3 GB 的模型：`epi_learning.tsfm` 會重播同一本筆記本先前用真正的 TimesFM 3.0 跑出並儲存的預測（`data/tsfm_cache/ph07_dengue.json`）。每個預測儲存格都會印出它走的是哪一條路徑。

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    sys.path.insert(0, '/content/python4epi/src')
    !pip install -q -e .

In [ ]:
# Colab 額外步驟：安裝 TimesFM 3.0（約 2 分鐘；權重稍後才下載，約 1.3 GB）
# 在你自己的電腦上用 uv：  uv sync --group timesfm
import sys
if 'google.colab' in sys.modules:
    !pip install -q "timesfm[torch]>=3.0.2"

## 步驟 1 — 每週登革熱序列

每一列是一個**流行病學週（morbidity week）**（ISO 週，週一到週日），從 2019-W01 到 2026-W39，包含依發病週計算的登革熱病例數，以及市氣象站的雨量。我們把病例數存成一個單純的 NumPy 陣列 `y`：預測模型要的是一串依時間順序排列的數字。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -- CJK 字型設定（避免中文標籤顯示成方框 □□□）--
import pathlib
import matplotlib.font_manager as fm

for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

# 動態偵測實際註冊的 CJK 字型名稱（避免 .ttc face 0 陷阱）
_discovered = []
for _entry in fm.fontManager.ttflist:
    _nlower = _entry.name.lower()
    if any(_kw in _nlower for _kw in ("cjk", "wenquanyi", "wqy")):
        if _entry.name not in _discovered:
            _discovered.append(_entry.name)
_preferred = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
_cjk_fonts = list(_discovered)
for _n in _preferred:
    if _n not in _cjk_fonts:
        _cjk_fonts.append(_n)
plt.rcParams["font.sans-serif"] = _cjk_fonts + [
    f for f in plt.rcParams.get("font.sans-serif", []) if f not in _cjk_fonts
]
plt.rcParams["axes.unicode_minus"] = False
del _discovered, _preferred, _cjk_fonts

from epi_learning import tsfm

weekly = pd.read_csv("data/synthetic/ph/santa_luz_dengue_weekly.csv", parse_dates=["week_start"])
y = weekly["cases"].to_numpy(dtype=float)          # 要預測的序列
rain = weekly["rainfall_mm"].to_numpy(dtype=float)  # 可能有幫助的輔助序列（共變數）
week_label = weekly["morbidity_week"].to_numpy()
row_of = {label: i for i, label in enumerate(week_label)}  # "2026-W31" -> 列號

print(f"{len(weekly)} 週：{week_label[0]} 到 {week_label[-1]}")
print(weekly.tail(6)[["morbidity_week", "cases", "rainfall_mm"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(weekly["week_start"], y, color="#0038A8", linewidth=1.4)
ax.axvspan(pd.Timestamp("2026-07-13"), weekly["week_start"].iloc[-1], color="#CE1126", alpha=0.12,
           label="颱風 Amihan 之後")
ax.set_ylabel("每週登革熱病例數")
ax.set_title("Santa Luz 市（合成資料）：每週登革熱病例數，2019-W01 至 2026-W39")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 步驟 2 — 先做基準線

在使用任何聰明的模型之前，Maya 請 Tala 做出一位疲憊的 DSO 在紙上就能做出的兩個最簡單的預測：

> **問問 Tala：** *我有一個 NumPy 陣列 `y`，是按時間順序排列的每週登革熱病例數（虛構範例：`[5, 9, 11, 4, ...]`，長度超過 52 週）。對於在第 `o` 列所做的預測（只知道 `y[:o]`），我想用兩種基準線預測接下來的 `H = 8` 週：(1)「最後觀測值」——把 `y[o-1]` 重複八次；(2)「季節性天真預測」——複製去年同樣的八週，`y[o-52 : o-52+8]`。請對一串起點都做一遍，並給我對照真實值的平均絕對誤差（MAE）。再加一個我可以手算驗證的檢查。*

Tala 的答案就是下面的儲存格。**起點**是從 2023-W01 到 2026-W31，每隔一週：共 94 次預測，每次 8 週長，而且每次都只用過去的資料。這就是**滾動起點回測（rolling-origin backtest）**：我們假裝站在每一個起點，做預測，然後才去看實際發生了什麼。

In [ ]:
H = 8                                             # 預測期間：8 週
first, last = row_of["2023-W01"], row_of["2026-W39"] + 1
origins = list(range(first, last - H + 1, 2))     # 每隔一週；每次預測最晚到 2026-W39 結束
actual = np.stack([y[o:o + H] for o in origins])  # 形狀 (94, 8)：實際發生的情況

last_value = np.stack([np.repeat(y[o - 1], H) for o in origins])
seasonal_naive = np.stack([y[o - 52:o - 52 + H] for o in origins])  # 2020 年之後沒有 ISO 第 53 週


def mae(pred, truth=actual):
    # 平均絕對誤差：平均每次差多少，單位是每週病例數
    return float(np.mean(np.abs(np.asarray(pred) - np.asarray(truth))))


print(f"{len(origins)} 個起點，第一次預測從 {week_label[origins[0]]} 開始，"
      f"最後一次從 {week_label[origins[-1]]} 開始")
print(f"MAE 最後觀測值：    {mae(last_value):.2f} 例/週")
print(f"MAE 季節性天真預測：{mae(seasonal_naive):.2f} 例/週")

**檢查一下。** 絕不要相信一個你沒有戳過的回測。我們取一個起點，手動檢查：(a)「最後觀測值」預測真的重複起點前一週；(b)「季節性天真預測」真的複製一年前同樣的週別；(c) 沒有任何預測用到它自己起點當週或之後的資料。

In [ ]:
k = 10                                   # 任何一個起點都可以
o = origins[k]
print("預測是在這一週之後做的：", week_label[o - 1], "| 預測的第一週：", week_label[o])
print("最後觀測值重複", y[o - 1], "->", last_value[k][:3], "...")
print("季節性天真預測複製", week_label[o - 52], "...", week_label[o - 52 + H - 1], "->", seasonal_naive[k])

assert np.all(last_value[k] == y[o - 1])
assert week_label[o - 52][5:] == week_label[o][5:], "一年前同樣的 ISO 週"
assert np.array_equal(actual[k], y[o:o + H])
assert all(len(c) == o for c, o in zip(tsfm.rolling_origin_contexts(y, origins), origins)), "歷史資料必須停在起點"
print("所有檢查都通過。")

## 步驟 3 — 載入 TimesFM 3.0

TimesFM 是一個**時間序列基礎模型（time-series foundation model）**：Google 用規模非常龐大的時間序列集合訓練過它一次，而我們以**零樣本**（zero-shot）的方式使用它——不用我們的資料再訓練，只要說「這是歷史資料，請給我接下來 8 週」。它會回傳中位數預測與九個**分位數**（0.1 … 0.9）；0.1 與 0.9 分位數圍出一個 **80% 預測區間**。

官方的 API 只有三行（`ModelConfig`、`TimesFM3Evaluator`、`predict_batch`）。如果沒有安裝 `timesfm`，這個儲存格會改用快取，並明說這件事。

In [ ]:
CACHE = tsfm.cache_path("ph07_dengue")   # data/tsfm_cache/ph07_dengue.json
MODE = tsfm.resolve_mode()               # 已安裝 timesfm 就是 "model"，否則是 "cache"

if MODE == "cache":
    model = None
    print("路徑 = CACHE：這個工作階段不執行 TimesFM；預測是從一次真實執行的結果重播而來。")
    print(tsfm.describe_cache(CACHE))
else:
    from timesfm3 import ModelConfig, TimesFM3Evaluator

    config = ModelConfig(
        checkpoint_path="google/timesfm-3.0-pytorch",  # Hugging Face 上的權重（約 1.3 GB，只在第一次執行時下載）
        per_core_batch_size=32,                        # 每次前向運算處理的序列數
        device=tsfm.default_device(),                  # GPU 執行環境用 "cuda"，否則用 "cpu"
    )
    model = TimesFM3Evaluator(config)
    print("路徑 = MODEL：TimesFM 3.0 已載入到", tsfm.default_device())

## 步驟 4 — 用 TimesFM 做同樣的回測

完全一樣的 94 個起點、一樣的 8 週預測期間、一樣的 MAE。`tsfm.rolling_origin_contexts(y, origins)` 為每個起點回傳 `y[:o]`——也就是允許模型看到的歷史資料。有模型時，`tsfm.forecast_batch` 只是呼叫 `model.predict_batch(contexts, horizon=8, return_quantiles=True)`；在 CPU 上這需要幾分鐘。

我們把兩個時期分開計分：**平常年份 2023–2025**，以及颱風那一年 **2026**。只看一個平均會把差異藏起來。

In [ ]:
contexts = tsfm.rolling_origin_contexts(y, origins)
bt = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_cases", cache_file=CACHE)
tfm = bt.forecast                                   # 中位數預測，形狀 (94, 8)
lo, hi = bt.quantile(0.1), bt.quantile(0.9)         # 80% 預測區間

year_of_end = weekly["iso_year"].to_numpy()[[o + H - 1 for o in origins]]
periods = {"2023-2025": year_of_end <= 2025, "2026": year_of_end == 2026}

rows = []
for period, sel in periods.items():
    for name, pred in [("最後觀測值", last_value), ("季節性天真預測", seasonal_naive), ("TimesFM 3.0", tfm)]:
        err = np.abs(pred[sel] - actual[sel])
        rows.append({"期間": period, "方法": name, "起點數": int(sel.sum()),
                     "MAE": err.mean(), "MAE 第1週": err[:, 0].mean(), "MAE 第8週": err[:, -1].mean()})
    rows.append({"期間": period, "方法": "TimesFM 80% 區間涵蓋率",
                 "起點數": int(sel.sum()), "MAE": tsfm.coverage(actual[sel], lo[sel], hi[sel])})
score = pd.DataFrame(rows)
print(score.round(2).to_string(index=False))

### 讀懂計分板

- **平常年份（2023–2025，75 次預測）：** TimesFM 的 MAE 是每週 **3.50** 例，季節性天真預測是 **4.31**，最後觀測值是 **4.76**。它的 80% 區間涵蓋了真實值的 **79%**——正是一個誠實的 80% 區間該有的樣子。完全不用訓練，這已經是一個有用的預測工具。
- **2026 年（19 次在 2026 年結束的預測）：** TimesFM 的 MAE 是 **5.32**，季節性天真預測是 **5.21**。在這裡，模型**並沒有比照抄去年更好**。它預測下一週最準（第 1 週 MAE 1.97），但到第 8 週就落後了（10.08，季節性天真預測是 8.68）。區間只涵蓋了 **72%**。
- 教訓：只看一個整體分數會把這件事藏起來。一定要把異常時期單獨拿出來評分——那正是地方衛生委員會最在意的時期。

## Tala 搞錯了：已經偷看過答案的預測

Maya 要求「更好的基準線：所有年份同一週的平均」。Tala 的程式碼得分漂亮極了——MAE 遠低於上面所有方法。Doc Mon 看了十秒鐘，問：*「這個平均用了哪幾年？」* 全部——**包括被預測的那幾週本身**。這就是**資料洩漏（leakage）**：預測被偷偷給看了答案。下面的儲存格重現了 Tala 的版本，接著是誠實的版本，只平均起點**之前**的年份。

另一個比較隱微的自欺方式，是只報告**最好**的那一週或最好的起點。我們也把它印出來，放在誠實的平均旁邊。

In [ ]:
iso_week = weekly["iso_week"].to_numpy()
iso_year = weekly["iso_year"].to_numpy()

# Tala 的版本（錯誤）：所有年份同一 ISO 週的平均，連測試的年份也包含在內
all_years_mean = weekly.groupby("iso_week")["cases"].mean()
leaky = np.stack([all_years_mean.loc[iso_week[o:o + H]].to_numpy() for o in origins])

# 誠實的版本：只用起點所在年份之前的三個完整年份
honest = []
for o in origins:
    past = weekly[(iso_year >= iso_year[o] - 3) & (iso_year < iso_year[o])]
    honest.append(past.groupby("iso_week")["cases"].mean().reindex(iso_week[o:o + H]).to_numpy())
honest = np.stack(honest)

print(f"同週平均，所有年份（有洩漏）：MAE {mae(leaky):.2f}  <- 看起來很棒，毫無意義")
print(f"同週平均，只用過去 3 年：    MAE {mae(honest):.2f}")
print(f"TimesFM 3.0（只看過去）：      MAE {mae(tfm):.2f}")

per_origin = np.abs(tfm - actual).mean(axis=1)
print(f"\n只挑好的來報：TimesFM 最好的起點 MAE 是 {per_origin.min():.2f} "
      f"（{week_label[origins[int(per_origin.argmin())]]}），最差的是 {per_origin.max():.2f} "
      f"（{week_label[origins[int(per_origin.argmax())]]}）；誠實的平均是 {per_origin.mean():.2f}。")

## 步驟 5 — 最難的時刻：疫情起飛

最重要的就是曲線轉彎的那幾週。我們給 TimesFM 到 2026-W30、W31、W32 和 W34 為止的歷史資料，每次請它預測接下來 8 週，並把預測疊在實際發生的情況上。（2026-W32 就是登革熱在[第 06 章](../06_signals_and_thresholds.md)第一次越過流行閾值的那一週。）

In [ ]:
takeoff_weeks = ["2026-W30", "2026-W31", "2026-W32", "2026-W34"]
takeoff_origins = [row_of[w] + 1 for w in takeoff_weeks]       # 歷史資料包含那一週
take = tsfm.forecast_batch(model, tsfm.rolling_origin_contexts(y, takeoff_origins), H,
                           cache_key="takeoff", cache_file=CACHE)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), sharey=True)
show = slice(row_of["2026-W20"], len(y))
for ax, k, w in zip(axes, range(4), takeoff_weeks):
    o = takeoff_origins[k]
    steps = np.arange(o, o + H)
    ax.bar(np.arange(len(y))[show], y[show], color="#9bbaff", label="通報病例數")
    ax.plot(steps, take.forecast[k], color="#CE1126", marker="o", ms=3, label="TimesFM 中位數")
    ax.fill_between(steps, take.quantile(0.1)[k], take.quantile(0.9)[k], color="#CE1126", alpha=0.18,
                    label="80% 區間")
    ax.axvline(o - 0.5, color="#1c1b29", linestyle="--", linewidth=1)
    ax.set_title(f"已知資料到 {w}", fontsize=10)
    ax.set_xticks([row_of["2026-W24"], row_of["2026-W32"], row_of["2026-W39"]], ["W24", "W32", "W39"])
    known = min(H, len(y) - o)
    inside = tsfm.coverage(y[o:o + known], take.quantile(0.1)[k][:known], take.quantile(0.9)[k][:known])
    print(f"到 {w} 為止：接下來 4 週的中位數 {np.round(take.forecast[k][:4]).astype(int)}，"
          f"實際 {y[o:o + 4].astype(int)}，實際值落在 80% 區間內：{known} 週中的 {inside:.0%}")
axes[0].set_ylabel("每週登革熱病例數")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 步驟 6 — 加入雨量有幫助嗎？

在這份合成序列裡，雨量**領先**病例：當雨量往後移 7 週時，相關性最高（見資料的 README）。TimesFM 3.0 接受**共變數**。`past_only_covariates` 提供它到起點為止的雨量——從 W31 做預測時，颱風 Amihan（2026-W29）的雨已經在歷史資料裡了。模型有用到它嗎？同樣的回測、同樣的起飛起點，這次加上雨量。

In [ ]:
rain_past = [rain[None, :o] for o in origins]            # 每個起點一個 (1, T) 陣列
bt_rain = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_rain", cache_file=CACHE,
                              past_only_covariates=rain_past)
take_rain = tsfm.forecast_batch(model, tsfm.rolling_origin_contexts(y, takeoff_origins), H,
                                cache_key="takeoff_rain", cache_file=CACHE,
                                past_only_covariates=[rain[None, :o] for o in takeoff_origins])

for period, sel in periods.items():
    print(f"{period}：MAE 只用病例數 {mae(tfm[sel], actual[sel]):.2f} | "
          f"加入雨量 {mae(bt_rain.forecast[sel], actual[sel]):.2f}")
k = takeoff_weeks.index("2026-W31")
print("\n從 2026-W31 起，接下來 8 週")
print("  實際值          ", y[takeoff_origins[k]:takeoff_origins[k] + H].astype(int))
print("  只用病例數      ", np.round(take.forecast[k]).astype(int))
print("  加入雨量        ", np.round(take_rain.forecast[k]).astype(int))

## 步驟 7 — Doc Mon 要的預測：2026-W40 到 W47

現在是真正的預測：用到 2026-W39 為止的全部歷史資料，預測接下來 8 週，並附上 80% 區間。但是請記得[第 06 章](../06_signals_and_thresholds.md)：**W39 是不完整的**——在 9 月 30 日資料凍結時，那一週發病的病例還在陸續通報中。所以我們也把模型用在**不含 W39** 的歷史資料上，再比較。如果兩者差很多，就代表最近幾週正在左右預測，你應該把這件事說出來。

In [ ]:
if model is not None:
    # 真正的 TimesFM 3.0 呼叫，只做一次，不經過輔助函式：
    out = next(model.predict_batch([y.astype(np.float32)], horizon=H, return_quantiles=True))
    print("predict_batch -> forecast", out.forecast.shape, "quantiles", out.quantiles.shape)

now = tsfm.forecast_batch(model, [y, y[:-1]], H, cache_key="forecast_now", cache_file=CACHE)
future_weeks = [f"2026-W{w}" for w in range(40, 48)]
table = pd.DataFrame({
    "週別": future_weeks,
    "下限 (q0.1)": now.quantile(0.1)[0],
    "中位數": now.forecast[0],
    "上限 (q0.9)": now.quantile(0.9)[0],
    "排除 W39 時的中位數": now.forecast[1][1:].tolist() + [np.nan],
}).round(0)
print(table.to_string(index=False))
print(f"\n用到 W38 為止的資料，模型預期 W39 有 {now.forecast[1][0]:.0f} 例；"
      f"到資料凍結時已通報 {y[-1]:.0f} 例。")
print(f"每週中位數加總，W40-W47：{now.forecast[0].sum():.0f} 例")
print(f"去年同樣的 8 週（2025-W40 到 W47）：{y[len(y) - 52:len(y) - 52 + H].sum():.0f} 例")

fig, ax = plt.subplots(figsize=(10, 3.8))
past = np.arange(row_of["2026-W20"], len(y))
ax.bar(past, y[past], color="#9bbaff", label="通報病例數（W39 不完整）")
steps = np.arange(len(y), len(y) + H)
ax.plot(steps, now.forecast[0], color="#CE1126", marker="o", ms=3, label="TimesFM 中位數")
ax.fill_between(steps, now.quantile(0.1)[0], now.quantile(0.9)[0], color="#CE1126", alpha=0.18,
                label="80% 預測區間")
ax.plot(np.arange(len(y) - 1, len(y) - 1 + H), now.forecast[1], color="#2f7d32", linestyle="--",
        label="排除 W39 時的中位數")
ax.set_xticks([row_of["2026-W24"], row_of["2026-W32"], row_of["2026-W39"], len(y) + 7],
              ["W24", "W32", "W39", "W47"])
ax.set_ylabel("每週登革熱病例數")
ax.set_title("從 2026-W39 起的預測（合成資料）")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 換你試試

這些都用你手上已有的陣列——不用再呼叫模型，所以在網站上和 Colab 上都能跑。

1. **哪一個預測週數最難？** 為每種方法印出第 1…8 週各自的 MAE（提示：`np.abs(tfm - actual).mean(axis=0)` 會給你八個數字）。在 2026 年，TimesFM 是從哪一週開始失去它對季節性天真預測的領先？
2. **80% 區間誠實嗎？** 在 2023–2025 年，數一數真實值落在區間**上方**與**下方**的週數（提示：`(actual > hi)[sel].mean()`）。一個誠實的區間，兩邊各會錯失約 10%。
3. **是規劃，不是預言。** 利用 `table`，為地方衛生委員會寫一句話，給出一個範圍而不是單一數字，並指出一件會讓你修正預測的事。

In [ ]:
# 1. 依預測週數計算 MAE（起手式：TimesFM，2026 年的起點）
sel = periods["2026"]
print("TimesFM，2026 年，依往後第幾週的 MAE：", np.abs(tfm[sel] - actual[sel]).mean(axis=0).round(1))
# ... 現在對 seasonal_naive 也做一遍，再比較

# 2. 區間錯失（起手式）
sel = periods["2023-2025"]
print("真實值高於區間的比例，2023-2025：", round(float((actual > hi)[sel].mean()), 2))
# ... 那低於區間的呢？

## 在工作中使用之前：授權

- TimesFM 的**程式碼**是 Apache-2.0，但 **TimesFM 3.0 的模型權重**（`google/timesfm-3.0-pytorch`）受 **TimesFM Non-Commercial License v1.0** 約束：可用於研究、評估與教學，**不得商業使用，也不得用於正式營運**。請自己讀一讀：[Hugging Face 上的 TimesFM 3.0 授權條款](https://huggingface.co/google/timesfm-3.0-pytorch/blob/main/LICENSE)。
- **TimesFM 2.5** 的權重是 Apache-2.0。Google 透過 Google Cloud 提供 TimesFM 的商業與正式營運使用。
- 對政府的衛生單位來說，這代表：學習、用舊資料做回測與探索——像這本筆記本這樣——是可以的。把 3.0 的預測用在例行的早期預警系統、公開的公報或例行的營運決策上，就不行。**在任何正式營運使用之前，請先確認授權條款（以及你們單位的規定）。** 這不是法律建議。
- `data/tsfm_cache/ph07_dengue.json` 裡的快取預測，是依該授權產生的模型輸出，用於這份非商業的教學教材。

**你學到了什麼：** 在相信一個模型之前，先在公平的回測裡贏過基準線；把困難的幾週單獨計分；絕不讓預測看到答案；而且模型看不見颱風、新的血清型，或是還沒通報的病例。